# ✦ EvoPrint AI — Fine-tuning sur WikiArt
**Dataset**: huggan/wikiart | **Modèle base**: Stable Diffusion v1.5 | **Technique**: LoRA

> ⚠️ Aller dans **Runtime → Change runtime type → T4 GPU** avant de lancer

In [ ]:
# ── Cellule 1 : Installation ──
!pip install -q diffusers transformers accelerate datasets peft huggingface_hub bitsandbytes
print('✅ Dépendances installées')

In [ ]:
# ── Cellule 2 : Connexion HuggingFace ──
from huggingface_hub import login
import os; login(token=os.environ['HF_TOKEN'])
print('✅ Connecté à HuggingFace')

In [ ]:
# ── Cellule 3 : Charger WikiArt ──
from datasets import load_dataset
from torchvision import transforms
import torch

print('📦 Chargement WikiArt (streaming)...')
dataset = load_dataset('huggan/wikiart', split='train', streaming=True)

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Prendre 500 images pour commencer
samples = list(dataset.take(500))
print(f'✅ {len(samples)} images chargées depuis WikiArt')
print(f'   Exemple: style={samples[0]["style"]}, artist={samples[0]["artist"]}')

In [ ]:
# ── Cellule 4 : Charger modèle + LoRA ──
from diffusers import StableDiffusionPipeline, DDPMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model

DEVICE = 'cuda'
MODEL_BASE = 'runwayml/stable-diffusion-v1-5'

print('📥 Chargement Stable Diffusion v1.5...')
tokenizer    = CLIPTokenizer.from_pretrained(MODEL_BASE, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(MODEL_BASE, subfolder='text_encoder').to(DEVICE)
unet         = UNet2DConditionModel.from_pretrained(MODEL_BASE, subfolder='unet').to(DEVICE)
scheduler    = DDPMScheduler.from_pretrained(MODEL_BASE, subfolder='scheduler')

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['to_q','to_v','to_k','to_out.0'],
    lora_dropout=0.1, bias='none'
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()
print('✅ LoRA configuré')

In [ ]:
# ── Cellule 5 : Entraînement ──
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
import random, os

OUTPUT_DIR  = './evoprint-wikiart-lora'
MAX_STEPS   = 500
LR          = 1e-4
os.makedirs(OUTPUT_DIR, exist_ok=True)

optimizer = AdamW(unet.parameters(), lr=LR)

PROMPTS = [
    'an impressionist painting, {s}, oil on canvas, fine art print quality',
    'expressionist artwork, {s}, bold colors, museum quality, EvoPrint style',
    'wikiart style painting, {s}, professional, high detail, print ready',
]
SUBJECTS = ['landscape','portrait','cityscape','floral composition','abstract','ocean']

unet.train(); text_encoder.eval()
losses = []

for step in range(MAX_STEPS):
    sample = random.choice(samples)
    img = sample['image'].convert('RGB')
    pixel = transform(img).unsqueeze(0).to(DEVICE)

    prompt = random.choice(PROMPTS).format(s=random.choice(SUBJECTS))
    ids = tokenizer(prompt, padding='max_length', max_length=77, truncation=True, return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad():
        enc = text_encoder(ids).last_hidden_state

    noise = torch.randn_like(pixel)
    t = torch.randint(0, scheduler.num_train_timesteps, (1,), device=DEVICE)
    noisy = scheduler.add_noise(pixel, noise, t)
    pred  = unet(noisy, t, enc).sample
    loss  = F.mse_loss(pred.float(), noise.float())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if step % 50 == 0:
        print(f'Step {step}/{MAX_STEPS} | Loss: {sum(losses[-50:])/len(losses[-50:]):.4f}')

# Sauvegarder
unet.save_pretrained(OUTPUT_DIR)
print(f'\n✅ Modèle sauvegardé dans {OUTPUT_DIR}')

In [ ]:
# ── Cellule 6 : Tester ──
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(MODEL_BASE, torch_dtype=torch.float16).to(DEVICE)
pipe.unet.load_attn_procs(OUTPUT_DIR)

prompts = [
    'marketing poster for a coffee brand, impressionist wikiart style, warm colors',
    'business card design, expressionist art style, professional, print ready',
]

for i, p in enumerate(prompts):
    img = pipe(p, num_inference_steps=30, guidance_scale=7.5).images[0]
    img.save(f'{OUTPUT_DIR}/result_{i+1}.jpg')
    print(f'✅ result_{i+1}.jpg généré')

# Afficher dans Colab
from IPython.display import display
from PIL import Image
display(Image.open(f'{OUTPUT_DIR}/result_1.jpg'))

In [ ]:
# ── Cellule 7 : Publier sur HuggingFace ──
from huggingface_hub import HfApi

HF_USERNAME = input('Votre username HuggingFace: ')
REPO_ID     = f'{HF_USERNAME}/evoprint-wikiart-lora'

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID, repo_type='model')

print(f'🚀 Modèle publié!')
print(f'   https://huggingface.co/{REPO_ID}')
print(f'\n   Ajoutez dans back/.env:')
print(f'   EVOPRINT_MODEL_ID={REPO_ID}')